In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver

from selenium.webdriver import ActionChains

from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.support.ui import Select

from selenium.webdriver.common.keys import Keys

from bs4 import BeautifulSoup

import pandas as pd

from time import sleep

import datetime

from pandas import ExcelWriter

import os

import re




In [2]:
regulatorName = 'MY CBMAL'
print(f"Running {regulatorName} Web Scraping Tool v.1.10.0")

now=datetime.datetime.now()

filename= '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

# scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


    

Running MY CBMAL Web Scraping Tool v.1.10.0


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------


regdict={
    	# 'MY CBMAL 1': ['https://www.bnm.gov.my/regulations/fsp-directory'], 
		# 'MY CBMAL 2': ['https://www.bnm.gov.my/regulations/fsp-directory'], 
		# 'MY CBMAL 3': ['https://www.bnm.gov.my/regulations/fsp-directory'], 
		# 'MY CBMAL 4': ['https://www.bnm.gov.my/regulations/fsp-directory'], 
		# 'MY CBMAL 5': ['https://www.bnm.gov.my/regulations/fsp-directory'], 
		# 'MY CBMAL 6': ['https://www.bnm.gov.my/regulations/fsp-directory'], 
		# 'MY CBMAL 7': ['https://www.bnm.gov.my/-/list-of-approved-money-brokers'], 
		# 'MY CBMAL 8': ['https://www.bnm.gov.my/-/approved-insurance-brokers'], 
		# 'MY CBMAL 9': ['https://www.bnm.gov.my/-/approved-insurance-and-takaful-brokers'], 
		# 'MY CBMAL 10': ['https://www.bnm.gov.my/-/approved-takaful-brokers-specialised-'], 
		# 'MY CBMAL 11': ['https://www.bnm.gov.my/-/registered-adjusters'], 
		# 'MY CBMAL 12': ['https://www.bnm.gov.my/-/approved-financial-advisers'],
		'MY CBMAL 13': ['https://www.bnm.gov.my/-/approved-international-marine-aviation-and-transit-mat-insurance-brokers'],
		# 'MY CBMAL 14': ['https://www.bnm.gov.my/-/approved-islamic-financial-advisers'],
		# 'MY CBMAL 15': ['https://www.bnm.gov.my/-/approved-electronic-trading-platforms-etp-'],
		# 'MY CBMAL 16': ['https://www.bnm.gov.my/list-of-financial-holding-companies']
         }





Typology ={

            'MY CBMAL 1': ['Commercial Banks','//input[@type="checkbox" and @value="Commercial Bank"]'],
            'MY CBMAL 2': ['Islamic Banks','//input[@type="checkbox" and @value="Islamic Bank"]'],
            #'MY CBMAL 3': 'International Islamic Bank',
            'MY CBMAL 3': ['Digital Banks','//input[@type="checkbox" and @value="Digital Bank"]'],
            'MY CBMAL 4': ['Investment Bank','//input[@type="checkbox" and @value="Investment Bank"]'],
            #'MY CBMAL 5': 'Other Financial Institutions',
            'MY CBMAL 5': ['Development Financial Institutions','//input[@type="checkbox" and @value="Development Financial"]'],
            'MY CBMAL 6': ['Licensed Insurance Companies & Takaful Operators',
                           '//input[@type="checkbox" and @value="Life Insurance|General Insurance"]',
                           '//input[@type="checkbox" and @value="Takaful"]',
                           '//input[@type="checkbox" and @value="Reinsurance|Retakaful"]'],
             # Insurance Companies, Takaful Operators, Reinsurers & Retakaful
            'MY CBMAL 7': 'Approved Money Brokers',
            'MY CBMAL 8': 'Approved Insurance Brokers',
            'MY CBMAL 9': 'Approved Insurance and Takaful Brokers',
            'MY CBMAL 10': 'Approved Takaful Brokers (Specialised)',
            'MY CBMAL 11': 'Registered Adjusters',
            'MY CBMAL 12': 'Approved Financial Advisers',
            'MY CBMAL 13': 'Approved International Marine, Aviation and Transit (MAT) Insurance Brokers',
            'MY CBMAL 14': 'Approved Islamic Financial Advisers',
            'MY CBMAL 15': 'Approved Electronic Trading Platforms (ETP)',
            'MY CBMAL 16': 'List of Financial Holding Companies',
            
}



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')


malaysian_cities = [
    'Kuala Lumpur', 
    'George Town', 
    'Ipoh', 
    'Kuching', 
    'Johor Bahru', 
    'Putrajaya', 
    'Kota Kinabalu', 
    'Shah Alam', 
    'Malacca City',
    'Alor Setar', 
    'Miri',
    'Petaling Jaya',
    'Kuala Terengganu', 
    'Iskandar Puteri',
    'Seberang Perai',
    'Seremban', 
    'Subang Jaya',
    'Pasir Gudang',    
    'Kuantan',
    'Klang'
]




In [4]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()


In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty

    return sqldict


def extract_city(address):
    for city in malaysian_cities:
        if city.lower() in address.lower():
            return city
    return ''



In [6]:

for reg in regdict:
    print('Working with {}.'.format(reg))
    clinks=[]
    driver.get(regdict[reg][0])
    sleep(2)
    if int(reg.split(' ')[-1]) <7:
    
        xpath_list = Typology.get(reg)

        for xpath in xpath_list[1:]:
            #print(xpath)
            sleep(2)
            checkbox = driver.find_element(By.XPATH, xpath)
            driver.execute_script("arguments[0].click();", checkbox)
            print('---Click the Checkbox---')
            sleep(3)
        

        # Locate the select element, Select the last option, try to load more data once. download the page time
        select_element = Select(driver.find_element("id", "dt-length-0"))
        options = select_element.options
        select_element.select_by_index(len(options) - 1)

        sleep(2)
        
        next_button = driver.find_element(By.XPATH, '//button[@data-dt-idx="next"]')

        # Check if it's disabled via class or aria-disabled
        is_disabled = "disabled" in next_button.get_attribute("class") or next_button.get_attribute("aria-disabled") == "true"

        if is_disabled:
            print("Next button is disabled.")
        else:
            print("Next button is enabled and clickable.")

        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find('table')
        inner_links = table.find_all('a')
        for inner_link in inner_links:
            clinks.append(inner_link['href'])
        print(f'Total Amount: {len(clinks)}')
        print('')

        for link in clinks:
            driver.get(link)
            sleep(2)
            inner_soup = BeautifulSoup(driver.page_source, 'html.parser')
            sleep(2)
            content = inner_soup.find('section', id='content') 
            sleep(2)
            name = content.find('div', class_='container').find('h2').text.strip()
            container = content.find('div', class_='container')
            print(name)
            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)   
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg][0])
            info_table = container.find('table')
            rows = info_table.find_all('tr')
            for row in rows:
                #address_, phone_, website_, fax_,facsimile_= '', '', '', '', ''
                if 'HQ Address' in row.text:
                    address_ = row.find_all('td')[1].text.strip()
                    address_ = address_ if len(address_) > 3 else ''
                    sqldict['Address_1'].append(address_.replace('\n',' '))
                    #print('Address:', address_)
                elif 'Telephone' in row.text:
                    phone_ = row.find_all('td')[1].text.strip()
                    phone_ = phone_ if len(phone_) > 3 else ''
                    sqldict['Phone'].append(phone_.replace('\n',' '))
                    #print('Phone:', phone_)
                elif 'Website' in row.text:
                    website_ = row.find_all('td')[1].text.strip()
                    website_ = website_ if len(website_) > 3 else ''
                    sqldict['Website'].append(website_.replace('\n',' '))
                    #print('Website:',website_)
                elif 'Fax' in row.text or 'Facsimile' in row.text:
                    fax_ = row.find_all('td')[1].text.strip()
                    fax_ = fax_ if len(fax_) > 3 else ''
                    sqldict['Fax'].append(fax_.replace('\n',' '))
                    #print('fax:',fax_)
            sqldict = bourange_same_length_array(sqldict)

    elif int(reg.split(' ')[-1]) == 7:
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find('table')
        trs = table.find('tbody').find_all('tr')
        for tr in trs:
            tds = tr.find_all('td')
            if tds and tds[0].text.strip().isdigit():
                row_data = [td.text.strip() for td in tds]
                name = row_data[1]
                address_ = row_data[3].split('\n')[0].split(':')[-1].strip()
                phone_ =  row_data[3].split('\n')[1].strip().split(':')[-1].strip()
                if 'Website' in row_data[3].split('\n')[-1]:
                    website_  = row_data[3].split('\n')[-1].split(' ')[-1].strip()
                    sqldict['Website'].append(website_)
                elif 'Facsimile' in row_data[3].split('\n')[-1]:
                    fax_ = row_data[3].split('\n')[-1].split(':')[-1].strip()
                    sqldict['Fax'].append(fax_)

                sqldict['Name'].append(name)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['Address_1'].append(address_) 
                sqldict['Phone'].append(phone_)
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict = bourange_same_length_array(sqldict)
    elif int(reg.split(' ')[-1]) == 8 or int(reg.split(' ')[-1]) == 9 or int(reg.split(' ')[-1]) == 10 or int(reg.split(' ')[-1]) == 11 or int(reg.split(' ')[-1]) == 12 or int(reg.split(' ')[-1]) == 14 :
            sleep(2)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            ent_list = soup.find('div', {'class','article-content page-content container'}).find('ol')
            if ent_list is not None:#case of ordered list (ol) with or contact information
                entities = ent_list.find_all('li')
                entities = [ent_.text.strip() for ent_ in entities]
                for entity in entities:
                    name = entity.split('\n')[0]
                    # print(name)
                    sqldict['Name'].append(name)
                    
                    entities = [ent_.strip() for ent_ in entity.split('\n')[1:]]
                    address_ = ','.join(entities[:])
                    # print(address_)
                    sqldict['Address_1'].append(address_)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict["ListProcessDate"].append(processdate)
                    sqldict["RegulationType"].append('Regulated')
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict = bourange_same_length_array(sqldict)
    elif int(reg.split(' ')[-1]) == 13 :
            sleep(2)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            ent_list = soup.find('div', {'class','article-content page-content container'}).find('table')
            if ent_list is not None:#case of ordered list (ol) with or contact information
                entities = ent_list.find_all('tr')
                for entity in entities[1:]:
                    infos = entity.find_all('td')
                    name = infos[1].find('b').text
                    address_ = infos[1].text.replace(name,'').replace('\t','').strip().replace('\n',' ')
                    sqldict['Name'].append(name)
                    sqldict['Address_1'].append(address_)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict["ListProcessDate"].append(processdate)
                    sqldict["RegulationType"].append('Regulated')
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict = bourange_same_length_array(sqldict)

    elif int(reg.split(' ')[-1]) == 15:
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find('table')
        trs = table.find('tbody').find_all('tr')
        for tr in trs:
            tds = tr.find_all('td')
            if tds and tds[0].text.strip().isdigit():
                row_data = [td.text.strip() for td in tds]
                name = row_data[1]
                etp_operator = row_data[2].strip()
                address_ = row_data[3].split('\n')[0].split(':')[-1].strip()
                phone_ =  row_data[3].split('\n')[1].strip().split(':')[-1].strip()
                if 'Website' in row_data[3].split('\n')[-1]:
                    website_  = row_data[3].split('\n')[-1].split(' ')[-1].strip()
                    sqldict['Website'].append(website_)
                elif 'Facsimile' in row_data[3].split('\n')[-1]:
                    fax_ = row_data[3].split('\n')[-1].split(':')[-1].strip()
                    sqldict['Fax'].append(fax_)
                

                sqldict['Name'].append(name)
                sqldict['Name - Mother Company'].append(etp_operator)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['Address_1'].append(address_) 
                sqldict['Phone'].append(phone_)
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict = bourange_same_length_array(sqldict)
    elif int(reg.split(' ')[-1]) == 16:
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        ent_list = soup.find('div', {'class','article-content page-content container'}).find('ol')
        if ent_list is not None:#case of ordered list (ol) with or contact information
            entities = ent_list.find_all('li')
            entities = [ent_.text.strip() for ent_ in entities]
            for entity in entities:
                if '***' not in entity:
                    sqldict['Name'].append(entity)
                    #print(entity)
                else:#name and address/zip
                    ent_split = [ele.strip() for ele in entity.split('***') if len(ele.strip())>1]
                    sqldict['Name'].append(ent_split[0])
                    #print(ent_split[0])
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict = bourange_same_length_array(sqldict)

        


            



    # driver.quit()

        

Working with MY CBMAL 13.


AttributeError: 'NoneType' object has no attribute 'text'

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

# df  = df.drop_duplicates()

df['Zip'] = df['Address_1'].apply(lambda x: re.search(r'(\d{5})(?!.*\d)', x).group(1) if re.search(r'(\d{5})(?!.*\d)', x) else None)

df['City'] = df['Address_1'].apply(extract_city)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_38216\735818324.py:15: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('list13.csv')

In [ ]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 24 values.
Key 'priority' has 24 values.
Key 'ListLabel' has 24 values.
Key 'Typology' has 24 values.
Key 'EntryType' has 24 values.
Key 'Name' has 24 values.
Key 'InternalID_1' has 24 values.
Key 'InternalID_1_type' has 24 values.
Key 'InternalID_2' has 24 values.
Key 'InternalID_2_type' has 24 values.
Key 'InternalID_3' has 24 values.
Key 'InternalID_3_type' has 24 values.
Key 'CoType' has 24 values.
Key 'License_Type' has 24 values.
Key 'Address_1' has 24 values.
Key 'Address_2' has 24 values.
Key 'City' has 24 values.
Key 'Zip' has 24 values.
Key 'Cntry' has 24 values.
Key 'Phone' has 24 values.
Key 'Fax' has 24 values.
Key 'Website' has 24 values.
Key 'Email' has 24 values.
Key 'RegulationType' has 24 values.
Key 'RegulationTypeCode' has 24 values.
Key 'RegulationDate' has 24 values.
Key 'CancellationDate' has 24 values.
Key 'RegCtry' has 24 values.
Key 'RegCode' has 24 values.
Key 'ListCode' has 24 values.
Key 'ListLanguage' has 24 values.
Key 'ListValidityDate' h

In [ ]:
for key, values in sqldict.items():
    print(f"Key '{key}' has values: {values}")

Key 'bvdid' has values: ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
Key 'priority' has values: ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
Key 'ListLabel' has values: ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
Key 'Typology' has values: ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
Key 'EntryType' has values: ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
Key 'Name' has values: ['Affin Bank Berhad', 'Alliance Bank Malaysia Berhad', 'AmBank (M) Berhad', 'Bangkok Bank Berhad', 'Bank of America Malaysia Berhad', 'Bank of China (Malaysia) Berhad', 'BNP Paribas Malaysia Berhad', 'China Construction Bank (Malaysia) Berhad', 'CIMB Bank Berhad', 'Citibank Berhad', 'Deutsche Bank (Malaysia) Berhad', 'Hong Leong Bank Berhad', 'HSBC Bank Ma